# 01 - Train a custom baseline (U-Net / MedNeXt / DA-VNet / AKS-UNet)

Runtime -> Change runtime type -> **T4 GPU**. Each team member sets `MODEL` to their model.
Training checkpoints to Drive every epoch; if Colab disconnects or units run out, **just re-run all cells** and it
resumes. Set `MAX_HOURS` a little below your remaining session so it stops cleanly.

In [ ]:
MODEL = "aksunet"
SMOKE = False
EPOCHS = 150
MAX_HOURS = 8

In [ ]:
%pip install -q monai

In [ ]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"

if not os.path.exists(REPO):
    subprocess.run([
        "git", "clone",
        "https://github.com/acm-research-f26/Radii.git",
        REPO
    ], check=True)

print("Repository ready:", os.path.exists(REPO))

In [ ]:
import os

DATA_ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"

CACHE = os.path.join(DATA_ROOT, "cache_0.9mm")
CLEAN = os.path.join(DATA_ROOT, "clean")
SPLITS = os.path.join(DATA_ROOT, "splits.json")

print("Cache:", os.path.exists(CACHE))
print("Clean:", os.path.exists(CLEAN))
print("Splits:", os.path.isfile(SPLITS))
print("Cache files:", len(os.listdir(CACHE)) if os.path.isdir(CACHE) else 0)
print("Clean files:", len(os.listdir(CLEAN)) if os.path.isdir(CLEAN) else 0)

In [ ]:

# --- Kaggle setup: dataset + GitHub repository ---

import os
import subprocess

# Kaggle dataset (read-only)
DATA_ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"

CACHE = os.path.join(DATA_ROOT, "cache_0.9mm")
CLEAN = os.path.join(DATA_ROOT, "clean")
SPLITS = os.path.join(DATA_ROOT, "splits.json")

# Writable project folder for training results
DRIVE = "/kaggle/working/figo"

# GitHub repository
REPO_URL = "https://github.com/acm-research-f26/Radii.git"
REPO = "/kaggle/working/figo-repo"

os.makedirs(DRIVE, exist_ok=True)

# Clone repository or update existing copy
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)

# Switch to repository
%cd {REPO}

# Install dependencies
!pip -q install -r requirements.txt

print("Kaggle setup complete!")
print("Dataset:", DATA_ROOT)
print("Training outputs:", DRIVE)


In [ ]:

# --- Week 2: Full training configuration ---

MODEL = "aksunet"
SMOKE = False
EPOCHS = 150
MAX_HOURS = 8
PATCH = "192 192 16"
BATCH = 2

CACHE = DATA_ROOT + "/cache_0.9mm"
SPLITS = DATA_ROOT + "/splits.json"

# Separate folder for full training
RUN = f"{DRIVE}/runs/full_{MODEL}"

os.makedirs(RUN, exist_ok=True)

print("Model:", MODEL)
print("Epochs:", EPOCHS)
print("Run folder:", RUN)
print("Cache exists:", os.path.isdir(CACHE))
print("Splits exists:", os.path.isfile(SPLITS))


In [ ]:
import subprocess

subprocess.run([
    "python", "-m", "figo.train",
    "--model", MODEL,
    "--cache-dir", CACHE,
    "--splits", SPLITS,
    "--out-dir", RUN,
    "--epochs", str(EPOCHS),
    "--max-hours", str(MAX_HOURS),
    "--patch-size", "192", "192", "16",
    "--batch-size", str(BATCH)
], cwd=REPO, check=True)

In [ ]:
# timing -> how many epochs fit the budget (row 1 = pure training, row 2 = training + validation)
import csv
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
for r in log:
    print('epoch', r['epoch'], r['epoch_sec'], 's')
if len(log) >= 3:
    t_train = float(log[1]['epoch_sec'])
    t_val = max(float(log[2]['epoch_sec']) - t_train, 0)
    per_epoch = t_train + t_val / 10          # validation runs every 10 epochs
    print(f'~{per_epoch:.0f} s per epoch on average; validation ~{t_val:.0f} s')
    for h in (6, 8, 10):
        print(f'  {h:2d} GPU-hours  ->  EPOCHS = {int(h * 3600 / per_epoch)}')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# learning curves
import csv, matplotlib.pyplot as plt
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
plt.plot([float(r['train_loss']) for r in log], label='train loss')
v = [(int(r['epoch']), float(r['val_mean'])) for r in log if r['val_mean']]
if v: plt.plot(*zip(*v), 'o-', label='val mean Dice')
plt.legend(); plt.xlabel('epoch'); plt.show()